# PML · Lecture 14 — PCA, step by step

Principal Component Analysis is **linear dimensionality reduction**: find a few orthogonal directions that capture most of the variation in your data. This notebook builds PCA **from scratch, one small step at a time** — center → covariance → eigen-decomposition → project → reconstruct → explained variance → choosing $k$ — proving each step on a tiny hand-checkable example, then running the finished thing on real data (iris and the 64-D digits).

Pure **`numpy` + `matplotlib`** (datasets from `scikit-learn`), no setup and no API key — run each cell top to bottom. Every worked number matches the lecture.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (5.6, 5.0)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
np.set_printoptions(precision=3, suppress=True)
print("ready — numpy", np.__version__)

## 1. The picture: data lives near a lower-dimensional subspace

Real data rarely *fills* its space — the features are correlated, so the cloud clusters near a thin subspace. PCA finds that subspace. Start with a **tilted 2-D cloud**: its two features move together, so the points stretch along one diagonal direction far more than the other. PCA's whole job is to *discover* that long direction.

In [ ]:
# a correlated 2-D cloud: stretch along one axis, rotate 30 degrees, shift off-origin
n = 300
base = rng.normal(size=(n, 2)) * np.array([3.0, 0.8])     # wide in x, narrow in y
theta = np.deg2rad(30)
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
X = base @ R.T + np.array([5.0, 4.0])                     # rotate, then shift away from origin

plt.scatter(*X.T, s=12, alpha=0.6)
plt.scatter(*X.mean(0), c='red', s=120, marker='X', zorder=5, label='mean')
plt.axis('equal'); plt.legend(); plt.title('a tilted, off-center 2-D cloud'); plt.show()
print("data mean:", X.mean(0), " <- note it is NOT at the origin")

## 2. Step 1 — center the data (and *why*)

PCA is about **variation around the mean**, so every version assumes mean-subtracted data: compute $\bar{x} = \frac1N\sum_n x_n$ and replace each point with $x_n - \bar{x}$. Skip this and the first component just points from the origin toward the cloud's *center* — capturing where the data *sits*, not how it *varies*. See both:

In [ ]:
mean = X.mean(axis=0)
Xc = X - mean                                             # centered: now the cloud sits at the origin

fig, ax = plt.subplots(1, 2, figsize=(10, 4.6))
ax[0].scatter(*X.T, s=10, alpha=0.5); ax[0].scatter(*mean, c='red', s=100, marker='X')
ax[0].axhline(0, lw=.8, c='k'); ax[0].axvline(0, lw=.8, c='k')
ax[0].set_title('raw: mean far from origin'); ax[0].axis('equal')
ax[1].scatter(*Xc.T, s=10, alpha=0.5); ax[1].scatter(0, 0, c='red', s=100, marker='X')
ax[1].axhline(0, lw=.8, c='k'); ax[1].axvline(0, lw=.8, c='k')
ax[1].set_title('centered: mean at the origin'); ax[1].axis('equal')
plt.tight_layout(); plt.show()
print("centered mean (~0):", Xc.mean(0))

## 3. Step 2 — the covariance matrix

How spread is the data along a unit direction $u$? Project each centered point to the scalar $u^\top x$; the **variance** of those projections is $u^\top S u$, where

$$S = \frac{1}{N}\sum_n x_n x_n^\top \quad (D\times D,\ \text{the covariance matrix}).$$

$S$ packs *all* the pairwise spread into one symmetric matrix. (Note the $1/N$ convention; numpy's `np.cov` divides by $N-1$, so its entries differ by a factor $N/(N-1)$.)

In [ ]:
S = (Xc.T @ Xc) / len(Xc)                                 # 1/N convention (lecture's)
print("covariance S =\n", S)
print("\ndiagonal = variance of each feature:", np.diag(S))
print("off-diagonal = how the two features co-vary:", S[0, 1],
      "(positive -> they rise together)")

# sanity: variance along a hand-picked direction u is u^T S u
u_test = np.array([1.0, 0.0])                             # the x-axis
print("\nvariance along x-axis  u^T S u =", u_test @ S @ u_test, "(= S[0,0])")

## 4. Step 3 — eigen-decomposition: the directions fall out

Maximizing $u^\top S u$ subject to $u^\top u = 1$ is a Lagrange-multiplier problem. Form $\mathcal{L}(u,\lambda) = u^\top S u - \lambda(u^\top u - 1)$; using $\nabla_u(u^\top S u) = 2Su$ (since $S$ is symmetric) and $\nabla_u(u^\top u) = 2u$, setting $\nabla_u\mathcal{L}=0$ gives $2Su - 2\lambda u = 0$, i.e.

$$S u = \lambda u.$$

So the best direction $u$ is an **eigenvector** of $S$ — and plugging back, the variance it captures is $u^\top S u = u^\top(\lambda u) = \lambda$, so its **eigenvalue** $\lambda$ *is* the variance along it (pick the largest to maximize). So:

| object | meaning in PCA |
|---|---|
| eigenvector $u_i$ of $S$ | the $i$-th principal component (a direction) |
| eigenvalue $\lambda_i$ | variance of the data **along** $u_i$ |
| largest $\lambda$ | the most-spread direction = PC1 |

$S$ is symmetric positive semi-definite → real non-negative eigenvalues, orthogonal eigenvectors. We sort them high → low and draw each PC as an arrow from the mean, scaled by $\sqrt{\lambda}$ (the standard deviation along it):

In [ ]:
evals, evecs = np.linalg.eigh(S)              # ascending, orthonormal columns
order = np.argsort(evals)[::-1]               # sort high -> low variance
evals, evecs = evals[order], evecs[:, order]
print("eigenvalues (variance per PC):", evals)
print("PC1 direction:", evecs[:, 0], "  PC2 direction:", evecs[:, 1])
print("PC1 . PC2 =", evecs[:, 0] @ evecs[:, 1], " (orthogonal)")

plt.scatter(*Xc.T, s=10, alpha=0.4)
for i in range(2):
    vec = evecs[:, i] * np.sqrt(evals[i]) * 2.5           # scale by sqrt(lambda) to see it
    plt.arrow(0, 0, *vec, width=0.05, color=['#d62728', '#2ca02c'][i],
              length_includes_head=True, zorder=5, label=f'PC{i+1} (lambda={evals[i]:.2f})')
plt.axis('equal'); plt.legend(); plt.title('principal components = eigenvectors of S'); plt.show()

### Prove it on a tiny set you can check by hand
The lecture's worked example: four centered points $(2,1),(-2,-1),(1,-1),(-1,1)$. By hand you get $S = \begin{bmatrix}2.5 & 0.5\\0.5 & 1.0\end{bmatrix}$, eigenvalues $\approx 2.65, 0.85$, and top eigenvector $\approx (0.958, 0.287)$. Reproduce it exactly:

In [ ]:
tiny = np.array([[2, 1], [-2, -1], [1, -1], [-1, 1.]])
print("mean (already 0):", tiny.mean(0))
S_tiny = (tiny.T @ tiny) / len(tiny)
print("S =\n", S_tiny, "  (expect [[2.5, 0.5], [0.5, 1.0]])")

ev, evec = np.linalg.eigh(S_tiny)
ev, evec = ev[::-1], evec[:, ::-1]
signs = np.where(evec[0] >= 0, 1.0, -1.0)      # flip so each PC points the +x way (sign is arbitrary)
evec = evec * signs
print("eigenvalues:", ev, "  (expect ~2.651, 0.849)")
print("top eigenvector:", evec[:, 0], "  (expect ~0.958, 0.287)")
# check against trace/determinant: lambda^2 - (trace)lambda + det = 0
print("trace:", np.trace(S_tiny), "= sum of eigenvalues", ev.sum())
print("det  :", np.linalg.det(S_tiny), "= product of eigenvalues", ev.prod())

## 5. The two views are the *same* answer

**View A** picks the direction of **maximum projected variance**. **View B** picks the subspace of **minimum reconstruction error** — project each point onto the line, and the little perpendicular segment it jumps is the error. These are identical because *total variance is fixed*: variance you keep (in the subspace) + variance you lose (perpendicular) = constant. Maximizing one minimizes the other. Reduce the cloud to $k=1$ and watch the error segments — they're all perpendicular to PC1:

In [ ]:
u1 = evecs[:, 0]                               # top PC of the big cloud
proj = (Xc @ u1)[:, None] * u1                 # project each centered point onto the PC1 line

plt.scatter(*Xc.T, s=10, alpha=0.4, label='original (centered)')
plt.scatter(*proj.T, s=12, c='#d62728', label='projected onto PC1')
for p, q in zip(Xc[::8], proj[::8]):           # a few reconstruction-error segments
    plt.plot([p[0], q[0]], [p[1], q[1]], c='gray', lw=0.7)
plt.axis('equal'); plt.legend(); plt.title('project to k=1: segments = reconstruction error'); plt.show()

# numerically: kept variance + lost variance = total variance
kept = np.var(Xc @ u1)                          # variance along PC1
total = np.sum(np.var(Xc, axis=0))              # total variance
lost = np.mean(((Xc - proj) ** 2).sum(1))       # mean squared perpendicular error
print(f"kept (var along PC1) = {kept:.3f}")
print(f"lost (recon error)   = {lost:.3f}")
print(f"kept + lost          = {kept + lost:.3f}   total variance = {total:.3f}  (equal)")

## 6. Step 4 — project and reconstruct

Keep the top $k$ eigenvectors as columns of $W$ ($D\times k$). A point's **reduced code** is $z = W^\top(x-\bar{x})$ ($k$ numbers); its **reconstruction** is $\hat{x} = Wz + \bar{x}$. The error $\lVert x-\hat{x}\rVert^2$ is exactly the part of $x$ along the *dropped* directions. The lecture projects $(2,1)$ onto the tiny set's top PC — reproduce $z\approx2.20$, $\hat{x}\approx(2.11,0.63)$, error $\approx0.15$:

In [ ]:
W = evec[:, :1]                                 # tiny set, k=1, top PC only
x = np.array([2.0, 1.0])                        # already centered (tiny set mean is 0)
z = W.T @ x                                      # 2. project -> code
x_hat = (W @ z)                                  # 3. reconstruct
err = ((x - x_hat) ** 2).sum()
print("code z        =", z, "  (expect ~2.20)")
print("reconstruction=", x_hat, "  (expect ~2.11, 0.63)")
print("squared error =", round(err, 3), "  (expect ~0.14; lecture's rounded by-hand gives 0.148)")

# the residual is parallel to the DISCARDED eigenvector u2 -- exactly the variance PCA drops
residual = x - x_hat
u2 = evec[:, 1]
print("\nresidual      =", residual)
print("discarded PC2 =", u2, " -> residual is parallel to it (that's the lost direction)")

## 7. Eigenvalues are a variance budget → choosing $k$

The eigenvalues sum to the **total variance**, so $\lambda_i / \sum_j \lambda_j$ is the **fraction of variance explained** by component $i$. To choose how many dimensions to keep, take the **cumulative** explained variance and stop at the smallest $k$ past your threshold. The lecture's example $\lambda = (6, 3, 0.8, 0.2)$: PC1 explains 60%, and $k=2$ reaches exactly 90%.

In [ ]:
lam = np.array([6.0, 3.0, 0.8, 0.2])
ratio = lam / lam.sum()
cum = np.cumsum(ratio)
print("per-component ratio:", ratio)
print("cumulative         :", cum)

# CHOOSING k: smallest k whose cumulative variance clears the threshold.
# Watch out: cum[1] is 0.6 + 0.3 = 0.8999999999999999 in floating point, so a naive
# `searchsorted(cum, 0.90)` would MISS the exact-90% case and return k=3. Use a tiny
# tolerance so "exactly 90%" counts as reaching 90% -- giving the correct k=2.
def choose_k(cum, threshold, tol=1e-9):
    if cum[-1] < threshold - tol:               # bar never reached -> keep everything
        return len(cum)
    return int(np.argmax(cum >= threshold - tol) + 1)

for thr in (0.90, 0.95, 0.98):
    print(f"k for {int(thr*100)}%: {choose_k(cum, thr)}")
# -> 90% at k=2 (the lecture's answer), 95% at k=3, 98% at k=3

### The scree plot: per-component bars + a cumulative line
Both diagnostics on one figure — bars for each $\lambda_i$ (the **elbow** is where they flatten) and a rising line for cumulative variance (keep the smallest $k$ past the threshold):

In [ ]:
fig, ax1 = plt.subplots(figsize=(6.2, 4.4))
idx = np.arange(1, len(lam) + 1)
ax1.bar(idx, ratio, color='#1f77b4', alpha=0.7, label='per-component')
ax1.set_xlabel('component'); ax1.set_ylabel('explained variance ratio'); ax1.set_xticks(idx)
ax2 = ax1.twinx()
ax2.plot(idx, cum, 'o-', color='#d62728', label='cumulative')
ax2.axhline(0.90, ls='--', color='gray'); ax2.set_ylabel('cumulative'); ax2.set_ylim(0, 1.05)
ax2.annotate('90%', (len(lam), 0.90), color='gray')
plt.title('scree plot: bars flatten at the elbow; line crosses 90% at k=2'); plt.show()

## 8. The stable route in practice — the SVD

In practice we rarely form $S$ (squaring the data worsens numerical conditioning). Instead take the **SVD** of the centered data matrix $X = U\Sigma V^\top$: the right-singular vectors (columns of $V$) **are** the principal components, and $\lambda_i = \sigma_i^2 / N$. This is what scikit-learn uses. Confirm both routes agree (components match up to a sign flip, which is arbitrary):

In [ ]:
# route 1: eigen-decomposition of S (what we did in step 3)
S_big = (Xc.T @ Xc) / len(Xc)
eval_eig, evec_eig = np.linalg.eigh(S_big)
eval_eig, evec_eig = eval_eig[::-1], evec_eig[:, ::-1]

# route 2: SVD of the CENTERED data matrix
U, sing, Vt = np.linalg.svd(Xc, full_matrices=False)
eval_svd = sing ** 2 / len(Xc)
evec_svd = Vt.T

print("eigh eigenvalues:", eval_eig)
print("svd  eigenvalues:", eval_svd, " (sigma^2 / N -- match)")
# align signs before comparing directions
aligned = evec_svd * np.sign((evec_svd * evec_eig).sum(0))
print("components agree (up to sign)?", np.allclose(aligned, evec_eig, atol=1e-6))

## 9. On real data — iris (project to 2-D and read the components)

PCA's most common everyday use: squash data to 2-D to *look* at it. Iris is 4-D (sepal/petal length/width); project it to its top 2 PCs and colour by species. The species separate cleanly even though PCA **never saw the labels** — the variance structure alone lines them up.

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()
A = iris.data
Ac = A - A.mean(0)

# standardize? iris features are all in cm, similar scale -> center is enough here.
Ua, sa, Vta = np.linalg.svd(Ac, full_matrices=False)
W2 = Vta[:2].T                                  # top 2 principal components (4 x 2)
Z = Ac @ W2                                      # project to 2-D (150 x 2)
ratio_iris = (sa ** 2) / (sa ** 2).sum()

for c in range(3):
    m = iris.target == c
    plt.scatter(Z[m, 0], Z[m, 1], s=14, label=iris.target_names[c])
plt.xlabel(f'PC1 ({ratio_iris[0]*100:.0f}% var)'); plt.ylabel(f'PC2 ({ratio_iris[1]*100:.0f}% var)')
plt.legend(); plt.title('iris projected onto its first two principal components'); plt.show()

print("PC1 loadings (how each feature weights into PC1):")
for name, w in zip(iris.feature_names, W2[:, 0]):
    print(f"  {w:+.3f}  {name}")
print("-> the biggest-magnitude loadings tell you what PC1 'means'.")

## 10. Choosing $k$ on real data — the 64-D digits

`load_digits` is 8×8 = **64-D** images of handwritten digits. How many components keep most of the variance? Compute the scree + cumulative curve and read off $k$ for 90% and 95% — far fewer than 64:

In [ ]:
from sklearn.datasets import load_digits
dig = load_digits()
D = dig.data
Dc = D - D.mean(0)
_, sd, Vtd = np.linalg.svd(Dc, full_matrices=False)
lam_d = sd ** 2 / len(Dc)
cum_d = np.cumsum(lam_d / lam_d.sum())

k90 = int(np.argmax(cum_d >= 0.90 - 1e-9) + 1)
k95 = int(np.argmax(cum_d >= 0.95 - 1e-9) + 1)
print(f"digits are {D.shape[1]}-D; keep k={k90} for 90% of variance, k={k95} for 95%")

fig, ax1 = plt.subplots(figsize=(6.4, 4.4))
ax1.bar(np.arange(1, 21), (lam_d / lam_d.sum())[:20], color='#1f77b4', alpha=0.7)
ax1.set_xlabel('component'); ax1.set_ylabel('explained variance ratio')
ax2 = ax1.twinx()
ax2.plot(np.arange(1, len(cum_d) + 1), cum_d, color='#d62728')
ax2.axhline(0.90, ls='--', color='gray'); ax2.set_ylabel('cumulative'); ax2.set_xlim(0, 64)
ax2.annotate('90%', (50, 0.91), color='gray')
plt.title(f'digits scree: 64-D, but k={k90} explains 90%'); plt.show()

### Reconstruction improves as $k$ grows
Reconstruct a digit from its top-$k$ code $\hat{x} = W z + \bar{x}$ for increasing $k$. The leftover error is exactly the sum of the *discarded* eigenvalues — add components and it falls, until extra ones stop helping:

In [ ]:
mean_d = D.mean(0)
sample = D[0]                                    # one digit image (64-vector)
fig, ax = plt.subplots(1, 6, figsize=(12, 2.4))
ax[0].imshow(sample.reshape(8, 8), cmap='gray'); ax[0].set_title('original'); ax[0].axis('off')
for j, k in enumerate([1, 4, 8, 16, 32], start=1):
    Wk = Vtd[:k].T
    z = Wk.T @ (sample - mean_d)
    recon = Wk @ z + mean_d
    ax[j].imshow(recon.reshape(8, 8), cmap='gray'); ax[j].axis('off')
    ax[j].set_title(f'k={k}')
plt.suptitle('PCA reconstruction sharpens as k grows'); plt.show()

# reconstruction error vs k == sum of discarded eigenvalues
errs = [lam_d[k:].sum() for k in range(1, 33)]
plt.figure(figsize=(6, 4))
plt.plot(range(1, 33), errs, 'o-'); plt.xlabel('k'); plt.ylabel('leftover variance (discarded lambdas)')
plt.title('error falls as k grows, then flattens'); plt.show()

## 11. Probabilistic PCA, in one breath

Everything above was geometric. **Probabilistic PCA** recasts it as a *generative* model — a latent linear-Gaussian: a low-D code $z\sim\mathcal{N}(0,I)$ maps up through a loading matrix $W$ plus isotropic noise, $x = Wz + \mu + \varepsilon$, $\varepsilon\sim\mathcal{N}(0,\sigma^2 I)$. Marginalizing $z$ gives $p(x)=\mathcal{N}(\mu,\ WW^\top + \sigma^2 I)$ — *low-rank-plus-noise*. The ML solution spans the **same** top-$k$ eigenvector subspace, with $\sigma^2$ = the **average of the discarded eigenvalues**. So PPCA isn't a different answer — it's PCA *with a probability story* (a likelihood for outlier scores, missing-data handling, and sampling new $x$). As $\sigma^2\to 0$ it collapses back to classical PCA.

> **One caution on the symbol $W$:** PPCA's loading $W$ spans the *same* subspace as §4's eigenvector matrix but is **not** the same matrix — its columns are scaled by $\sqrt{\lambda-\sigma^2}$ (not unit-norm) and are fixed only up to a rotation. And **factor analysis** generalizes PPCA by letting the noise be a *diagonal* $\Psi$ (a different variance per feature) instead of a single $\sigma^2$.

In [ ]:
# sigma^2 (ML) = average of the discarded eigenvalues -- demo on the digits at k=10
k = 10
sigma2 = lam_d[k:].mean()
print(f"PPCA noise sigma^2 at k={k}: {sigma2:.3f}  (= mean of the {len(lam_d)-k} discarded eigenvalues)")
print("as sigma^2 -> 0, PPCA's Cov(x) = W W^T + sigma^2 I collapses to rank-k W W^T = classical PCA")

## 12. Two cautions

- **PCA is scale-sensitive.** A feature in millimetres will dominate one in metres purely by unit choice. When features have different units, **standardize** (divide each by its standard deviation) before PCA.
- **PCA is linear.** It finds flat subspaces only; a curved manifold (e.g. points on a Swiss roll) needs nonlinear methods — kernel PCA, t-SNE, UMAP, or an autoencoder (Lecture 12).

In [ ]:
# scale sensitivity: blow up ONE feature's units and watch PC1 swing to chase it
A2 = load_iris().data.copy()
A2[:, 0] *= 100.0                                # pretend feature 0 is in different units
raw_pc1  = np.linalg.svd(A2 - A2.mean(0), full_matrices=False)[2][0]
Z2 = (A2 - A2.mean(0)) / A2.std(0)               # standardize first
std_pc1  = np.linalg.svd(Z2, full_matrices=False)[2][0]
print("PC1 on RAW (feature 0 blown up):", np.abs(raw_pc1).round(2), "-> dominated by feature 0")
print("PC1 after STANDARDIZING        :", np.abs(std_pc1).round(2), "-> balanced across features")

## Recap — the whole algorithm
1. **Center:** $x \leftarrow x - \bar{x}$.
2. **Covariance:** $S = \frac1N X_c^\top X_c$ (or skip it and SVD $X_c$ directly).
3. **Eigen-decompose:** $S u_i = \lambda_i u_i$; sort high → low. $u_i$ = PC$i$, $\lambda_i$ = variance along it.
4. **Keep top $k$:** $W = [u_1,\dots,u_k]$; project $z = W^\top(x-\bar{x})$, reconstruct $\hat{x} = Wz + \bar{x}$.
5. **Choose $k$:** smallest $k$ whose cumulative $\lambda$ clears your threshold (mind the floating-point edge at an *exact* threshold).

Maximizing kept variance = minimizing reconstruction error — one objective, two readings.